Connected to .venv01 (3.13.11) (Python 3.13.11)

In [ ]:
# =============================================================================================================
# ======================================= HYPERPARAMETER TUNING ===============================================
# =============================================================================================================

import os
import sys
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import optuna


sys.path.insert(0, "/Users/shah/CODE_BOOK_4/THESIS_WORKING/THESIS_2")
sys.path.insert(0, "/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/MFIV_calibration")
os.chdir("/Users/shah/CODE_BOOK_4/THESIS_WORKING/THESIS_2")


# =============================================================================================================
# ======================================= COMMON PARAMETERS ===================================================
# =============================================================================================================

YEARS = [2019, 2020, 2021]
QUARTERS = ["q1", "q2", "q3", "q4"]

SEQUENCE_LENGTH = 168
FORWARD_HOURS = 24
DELTA_TARGET = 0.20

VAL_FRACTION = 0.20
TRAIN_FRACTIONS = [0.40, 0.60, 0.80]

EPOCHS = 40
PATIENCE = 6
SEED = 42

N_TRIALS = 40


# =============================================================================================================
# ======================================= DATA PATHS ==========================================================
# =============================================================================================================

DATA_DIR = Path("../data/rawData")

OUTPUTS_PATH = (
    "/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/"
    "data/IS/outputs_by_year/outputs_by_year_01.pkl"
)

X_BY_YEAR_PATH = (
    "/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/"
    "data/IS/x_by_year_fut/x_by_year.pkl"
)

RFR_PATH = (
    "/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/"
    "data/IS/ready_data/d_btc_by_year_all.pkl"
)


# =============================================================================================================
# ======================================= TCN IMPORTS =========================================================
# =============================================================================================================

sys.modules.pop("srcB.tcn_model_v2", None)

from srcB.tcn_model_v2 import (
    build_tcn_features,
    build_forward_targets,
    make_training_sequences,
    SmallCausalTCN,
    _scalers,
    _train,
    _get_device,
)


# =============================================================================================================
# ======================================= LOAD IS DATA ========================================================
# =============================================================================================================

def load_is_data():

    with open(OUTPUTS_PATH, "rb") as f:
        outputs_by_year = pickle.load(f)

    X_by_year = pd.read_pickle(X_BY_YEAR_PATH)

    with open(RFR_PATH, "rb") as f:
        d_btc_by_year = pickle.load(f)

    d_btc_raw_by_year = {}
    df_raw_by_year = {}

    for y in YEARS:

        d_btc_raw_by_year[y] = pd.read_parquet(
            DATA_DIR / f"btc_underlying_minutely_{y}.parquet"
        )

        df_raw_by_year[y] = pd.read_parquet(
            DATA_DIR / f"derivative_data_{y}.parquet"
        )

        d_btc_raw_by_year[y] = d_btc_raw_by_year[y].set_index(
            pd.to_datetime(d_btc_raw_by_year[y]["date_time"])
        )

        df_raw_by_year[y] = df_raw_by_year[y].set_index(
            pd.to_datetime(df_raw_by_year[y]["date_time"])
        )

        d_btc_raw_by_year[y]["rfr"] = d_btc_by_year[y]["rfr"].reindex(
            d_btc_raw_by_year[y].index,
            method="ffill"
        )

    return (
        X_by_year,
        outputs_by_year,
        d_btc_raw_by_year,
        df_raw_by_year,
    )


# =============================================================================================================
# ======================================= PREPARE ONE TCN YEAR ================================================
# =============================================================================================================

def prepare_tcn_year(
    year,
    X_by_year,
    outputs_by_year,
    d_btc_raw_by_year,
    df_raw_by_year,
):

    X_df = X_by_year[year].copy().sort_index()

    X_df["rv"] = X_df["rt"] ** 2

    for H in [12, 24, 72, 168, 336]:
        X_df[f"MFIV_{H}"] *= H / 8760.0

    garch = pd.concat(
        [outputs_by_year[year][q]["garch"] for q in QUARTERS]
    ).sort_index()

    X_df["garch"] = garch.reindex(X_df.index)

    features = build_tcn_features(X_df)

    targets = build_forward_targets(
        option_df=df_raw_by_year[year],
        time_df=d_btc_raw_by_year[year],
        times=features.dropna().index,
        forward_hours=FORWARD_HOURS,
        delta_target=DELTA_TARGET,
    )

    X, y, idx = make_training_sequences(
        features,
        targets,
        sequence_length=SEQUENCE_LENGTH,
    )

    return {
        "X": X,
        "y": y,
        "idx": idx,
        "n_features": features.shape[1],
    }


# =============================================================================================================
# ======================================= PREPARE ALL TCN YEARS ===============================================
# =============================================================================================================

def prepare_tcn_data(
    X_by_year,
    outputs_by_year,
    d_btc_raw_by_year,
    df_raw_by_year,
):

    tcn_data = {}

    for y in YEARS:

        print("Preparing TCN data:", y)

        tcn_data[y] = prepare_tcn_year(
            y,
            X_by_year,
            outputs_by_year,
            d_btc_raw_by_year,
            df_raw_by_year,
        )

        print("Sequences:", len(tcn_data[y]["idx"]))

    return tcn_data


# =============================================================================================================
# ======================================= ROLLING CHRONOLOGICAL FOLD ==========================================
# =============================================================================================================

def make_rolling_fold(data, train_fraction):

    X = data["X"]
    y = data["y"]
    idx = data["idx"]

    val_start_i = int(len(idx) * train_fraction)
    val_end_i = int(len(idx) * (train_fraction + VAL_FRACTION))

    val_start = idx[val_start_i]

    train_mask = (
        idx
        < val_start - pd.Timedelta(hours=FORWARD_HOURS)
    )

    if val_end_i < len(idx):

        val_end = idx[val_end_i]

        val_mask = (
            (idx >= val_start)
            & (idx < val_end)
        )

    else:
        val_mask = idx >= val_start

    return {
        "X_train": X[train_mask],
        "y_train": y[train_mask],
        "X_val": X[val_mask],
        "y_val": y[val_mask],
    }


# =============================================================================================================
# ======================================= FIT ONE TCN FOLD ====================================================
# =============================================================================================================

def fit_tcn_fold(fold, n_features, params):

    np.random.seed(SEED)
    torch.manual_seed(SEED)

    X_train = fold["X_train"]
    y_train = fold["y_train"]

    X_val = fold["X_val"]
    y_val = fold["y_val"]

    x_mean, x_std, y_mean, y_std = _scalers(
        X_train,
        y_train
    )

    X_train = ((X_train - x_mean) / x_std).astype(np.float32)
    X_val = ((X_val - x_mean) / x_std).astype(np.float32)

    y_train = ((y_train - y_mean) / y_std).astype(np.float32)
    y_val = ((y_val - y_mean) / y_std).astype(np.float32)

    model = SmallCausalTCN(
        n_features=n_features,
        channels=params["channels"],
        kernel_size=params["kernel_size"],
        dropout=params["dropout"],
    )

    model, best_epoch, history = _train(
        model=model,
        X=X_train,
        y=y_train,
        device=_get_device(),
        epochs=EPOCHS,
        batch_size=params["batch_size"],
        learning_rate=params["learning_rate"],
        weight_decay=params["weight_decay"],
        X_val=X_val,
        y_val=y_val,
        patience=PATIENCE,
    )

    best_val_mse = history["val_loss"].min()

    return best_val_mse


# =============================================================================================================
# ======================================= TCN SEARCH SPACE ====================================================
# =============================================================================================================

def suggest_tcn_params(trial):

    channel_width = trial.suggest_categorical(
        "channel_width",
        [8, 16, 32]
    )

    num_blocks = trial.suggest_categorical(
        "num_blocks",
        [5, 6, 7]
    )

    kernel_size = trial.suggest_categorical(
        "kernel_size",
        [3, 5]
    )

    dropout = trial.suggest_categorical(
        "dropout",
        [0.0, 0.10, 0.20]
    )

    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-4,
        3e-3,
        log=True
    )

    weight_decay = trial.suggest_categorical(
        "weight_decay",
        [0.0, 1e-6, 1e-5, 1e-4, 1e-3]
    )

    batch_size = trial.suggest_categorical(
        "batch_size",
        [64, 128, 256]
    )

    channels = (channel_width,) * num_blocks

    return {
        "channel_width": channel_width,
        "num_blocks": num_blocks,
        "channels": channels,
        "kernel_size": kernel_size,
        "dropout": dropout,
        "learning_rate": learning_rate,
        "weight_decay": weight_decay,
        "batch_size": batch_size,
    }


# =============================================================================================================
# ======================================= CHECK TCN RECEPTIVE FIELD ===========================================
# =============================================================================================================

def valid_tcn_receptive_field(params):

    receptive_field = (
        1
        + 2
        * (params["kernel_size"] - 1)
        * (2 ** params["num_blocks"] - 1)
    )

    return receptive_field >= SEQUENCE_LENGTH


# =============================================================================================================
# ======================================= EVALUATE ONE TCN TRIAL ==============================================
# =============================================================================================================

def evaluate_tcn_trial(trial, tcn_data):

    params = suggest_tcn_params(trial)

    if not valid_tcn_receptive_field(params):
        raise optuna.TrialPruned()

    all_losses = []

    resources = [1, 2, 4]

    for resource, train_fraction in zip(resources, TRAIN_FRACTIONS):

        round_losses = []

        for y in YEARS:

            fold = make_rolling_fold(
                tcn_data[y],
                train_fraction
            )

            val_mse = fit_tcn_fold(
                fold=fold,
                n_features=tcn_data[y]["n_features"],
                params=params,
            )

            round_losses.append(val_mse)

        all_losses.extend(round_losses)

        mean_val_mse = float(np.mean(all_losses))

        trial.report(
            mean_val_mse,
            step=resource
        )

        if trial.should_prune():
            raise optuna.TrialPruned()

    return mean_val_mse


# =============================================================================================================
# ======================================= RUN TCN TUNING ======================================================
# =============================================================================================================

def tune_tcn(tcn_data):

    sampler = optuna.samplers.TPESampler(
        seed=SEED,
        n_startup_trials=10
    )

    pruner = optuna.pruners.SuccessiveHalvingPruner(
        min_resource=1,
        reduction_factor=2
    )

    study = optuna.create_study(
        direction="minimize",
        sampler=sampler,
        pruner=pruner
    )

    study.optimize(
        lambda trial: evaluate_tcn_trial(
            trial,
            tcn_data
        ),
        n_trials=N_TRIALS
    )

    return study

/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/MFIV_calibration/.venv01/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
if __name__ == "__main__":

    (
        X_by_year,
        outputs_by_year,
        d_btc_raw_by_year,
        df_raw_by_year,
    ) = load_is_data()

    tcn_data = prepare_tcn_data(
        X_by_year,
        outputs_by_year,
        d_btc_raw_by_year,
        df_raw_by_year,
    )

    tcn_study = tune_tcn(tcn_data)

    print("\nBEST TCN VALIDATION MSE")
    print(tcn_study.best_value)

    print("\nBEST TCN PARAMETERS")
    print(tcn_study.best_params)

    best_tcn_params = tcn_study.best_params.copy()

    best_tcn_params["channels"] = (
        best_tcn_params["channel_width"],
    ) * best_tcn_params["num_blocks"]

    print("\nTCN CHANNELS")
    print(best_tcn_params["channels"])

Preparing TCN data: 2019
Sequences: 2194
Preparing TCN data: 2020
Sequences: 4093
Preparing TCN data: 2021


[I 2026-09-02 19:35:56,634] A new study created in memory with name: no-name-e0e89c9a-0657-41dc-9eed-8d972d552162


Sequences: 5008


[I 2026-09-02 19:36:25,974] Trial 0 finished with value: 0.6713264650768704 and parameters: {'channel_width': 16, 'num_blocks': 5, 'kernel_size': 5, 'dropout': 0.1, 'learning_rate': 0.002708160864249967, 'weight_decay': 0.0, 'batch_size': 64}. Best is trial 0 with value: 0.6713264650768704.
[I 2026-09-02 19:36:41,782] Trial 1 pruned. 
[I 2026-09-02 19:37:10,311] Trial 2 finished with value: 0.6661134676800834 and parameters: {'channel_width': 16, 'num_blocks': 5, 'kernel_size': 5, 'dropout': 0.2, 'learning_rate': 0.0013962563737015762, 'weight_decay': 0.0, 'batch_size': 256}. Best is trial 2 with value: 0.6661134676800834.
[I 2026-09-02 19:38:00,808] Trial 3 pruned. 
[I 2026-09-02 19:38:19,310] Trial 4 pruned. 
[I 2026-09-02 19:38:29,472] Trial 5 pruned. 
[I 2026-09-02 19:38:43,469] Trial 6 pruned. 
[I 2026-09-02 19:38:43,472] Trial 7 pruned. 
[I 2026-09-02 19:40:00,884] Trial 8 pruned. 
[I 2026-09-02 19:40:00,900] Trial 9 pruned. 
[I 2026-09-02 19:40:08,420] Trial 10 pruned. 
[I 2026-


BEST TCN VALIDATION MSE
0.6661134676800834

BEST TCN PARAMETERS
{'channel_width': 16, 'num_blocks': 5, 'kernel_size': 5, 'dropout': 0.2, 'learning_rate': 0.0013962563737015762, 'weight_decay': 0.0, 'batch_size': 256}

TCN CHANNELS
(16, 16, 16, 16, 16)
